# Guía de Práctica 03: Aseguramiento de la Calidad de Software (SQA / ACS)
### Universidad Nacional de San Antonio Abad del Cusco (UNSAAC)
**Facultad de Ingeniería Informática, Sistemas, Redes y Telecomunicaciones**  
**Escuela Profesional de Ingeniería Informática y de Sistemas**  
**Curso:** Ingeniería de Software II | **Docente:** Mgt. Edwin Alvarez Mamani | **Semestre:** 2026-I  
**Caso Práctico 01:** TechBank SAC — Plataforma Web Transaccional (Versión Beta)

---
### Integrantes del Equipo de Laboratorio:
- **Pavel Alvaro Motta Mendoza** (Cód. 210937) - *Moderador SQA / Planificador*
- **Edmil Jampier Saire Bustamante** (Cód. 174449) - *Autor / Desarrollador Web*
- **Nayeli Constantina Labra Huaita** (Cód. 195048) - *Secretaria de Actas SQA*
- **Brayan Rodrigo Quispe Castillo** (Cód. 221986) - *Lector Técnico / Analista*
- **Joham Esau Quispe Huillca** (Cód. 211358) - *Revisor SQA / Tester*


In [ ]:
# ==============================================================================
# ACTIVIDAD 1: CARGA E INGESTA DEL DATASET DE INCIDENCIAS
# ==============================================================================
import pandas as pd
import numpy as np

# Carga del dataset base de incidencias
df_muestra = pd.read_csv('incidencias_software.csv')
print('=== DATASET BASE: incidencias_software.csv (10 registros iniciales) ===')
display(df_muestra)

print('\nResumen estadistico de tiempos de deteccion y reparacion:')
display(df_muestra[['tiempo_deteccion_hs', 'tiempo_reparacion_hs']].describe())


---
## ACTIVIDAD 2: CONSTRUCCIÓN DEL DIAGRAMA DE PARETO CON PYTHON
**Objetivo:** Identificar el 20% de las causas que generan el 80% de los defectos en TechBank SAC.  
Se analiza el dataset de 942 incidencias clasificadas en 8 categorías según la taxonomía SQA:
- **EEI:** Especificaciones Erróneas / Incompletas
- **Otros:** Diversos errores no agrupados
- **MCC:** Mala Comunicación Cliente-Desarrollador
- **ERD:** Error de Representación de Datos
- **IET:** Inconsistencia en Casos de Prueba
- **PLT:** Error de Traducción de Lenguaje / Código
- **ICI:** Interfaz Inconsistente
- **EDL:** Error de Diseño Lógico


In [ ]:
# ==============================================================================
# ACTIVIDAD 2: CONSTRUCCIÓN Y GRAFICACIÓN DEL DIAGRAMA DE PARETO
# ==============================================================================
import matplotlib.pyplot as plt

# Dataset extendido de frecuencia de errores
data = {
    'Causa_Error': ['EEI (Especif. Incompletas)', 'Otros (Diversos)', 
                    'MCC (Mala Comunicacion)', 'ERD (Error Repres. Datos)', 
                    'IET (Inconsist. Pruebas)', 'PLT (Error Traduc. Codigo)', 
                    'ICI (Interfaz Inconsist.)', 'EDL (Error Diseño Logico)'],
    'Frecuencia': [205, 193, 156, 130, 95, 60, 58, 45]
}

df_pareto = pd.DataFrame(data)
df_pareto = df_pareto.sort_values(by='Frecuencia', ascending=False).reset_index(drop=True)

# Cálculo del porcentaje individual y acumulado
df_pareto['Porcentaje'] = (df_pareto['Frecuencia'] / df_pareto['Frecuencia'].sum()) * 100
df_pareto['Acumulado'] = df_pareto['Porcentaje'].cumsum()
df_pareto['Clasificacion'] = df_pareto['Acumulado'].apply(lambda x: 'Vital (<= 80%)' if x <= 80 else 'Trivial (> 80%)')

print('=== TABLA DE FRECUENCIAS Y PORCENTAJES ACUMULADOS DE PARETO ===')
display(df_pareto)

# Graficación del Diagrama de Pareto (Doble Eje)
fig, ax1 = plt.subplots(figsize=(11, 6))

colores = ['#800000' if c == 'Vital (<= 80%)' else '#64748b' for c in df_pareto['Clasificacion']]
bars = ax1.bar(df_pareto['Causa_Error'], df_pareto['Frecuencia'], color=colores, edgecolor='black', alpha=0.9, width=0.65)
ax1.set_ylabel('Frecuencia Absoluta de Defectos', color='#800000', fontsize=11, fontweight='bold')
ax1.tick_params(axis='x', rotation=30, labelsize=9)
ax1.set_ylim(0, max(df_pareto['Frecuencia']) * 1.15)

for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 3, f'{int(yval)}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(df_pareto['Causa_Error'], df_pareto['Acumulado'], color='#b91c1c', marker='D', linewidth=2.2, markersize=7)
ax2.axhline(80, color='#b45309', linestyle='--', linewidth=1.8, label='Umbral Pareto 80%')
ax2.set_ylabel('Porcentaje Acumulado (%)', color='#b91c1c', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 105)

for i, txt in enumerate(df_pareto['Acumulado']):
    ax2.annotate(f'{txt:.1f}%', (i, df_pareto['Acumulado'][i] + 2), ha='center', fontsize=8.5, fontweight='bold', color='#991b1b')

plt.title('Diagrama de Pareto — Aseguramiento Estadistico de la Calidad (ACS)\nCaso TechBank SAC (n=942 incidencias)', fontsize=13, fontweight='bold', pad=15)
ax2.legend(loc='center right')
plt.grid(True, alpha=0.25, linestyle=':')
plt.tight_layout()
plt.show()

print('\n[DIAGNÓSTICO SQA]: Las 4 primeras causas vitales (EEI, Otros, MCC, ERD) acumulan el 72.7% de los defectos.')
print('La categoría prioritaria absoluta es EEI (21.8%, 205 defectos).')


---
## ACTIVIDAD 3: CÁLCULO DE MÉTRICAS DE CONFIABILIDAD Y DISPONIBILIDAD
**Objetivo:** Determinar cuantitativamente el TMEF, TMPF, TMPR y el porcentaje de Disponibilidad del sistema de software, evaluando si cumple con el SLA de banca en línea (99.95%).
- **TMPF:** Tiempo Medio Para Falla = 450 horas
- **TMPR:** Tiempo Medio Para Reparación = 6 horas


In [ ]:
# ==============================================================================
# ACTIVIDAD 3: CÁLCULO DE INDICADORES DE CONFIABILIDAD Y EVALUACIÓN DE SLA
# ==============================================================================
TMPF = 450.0  # horas de operacion continua antes de una falla
TMPR = 6.0    # horas promedio de reparacion / restablecimiento del servicio

# Calculo formal segun Pressman / IEEE Std 982.1
TMEF = TMPF  # Por definicion clasica en software
disponibilidad = (TMPF / (TMPF + TMPR)) * 100
indisponibilidad = 100 - disponibilidad

horas_caida_mes = (indisponibilidad / 100) * (24 * 30.5)
horas_caida_anio = (indisponibilidad / 100) * (24 * 365)

SLA_BANCA = 99.95
horas_sla_anio = ((100 - SLA_BANCA) / 100) * (24 * 365)
brecha_sla = disponibilidad - SLA_BANCA

print('================================================================')
print('        REPORTE CUANTITATIVO DE CONFIABILIDAD Y DISPONIBILIDAD  ')
print('================================================================')
print(f'TMPF (Tiempo Medio Para la Falla):         {TMPF:.2f} horas')
print(f'TMPR (Tiempo Medio Para la Reparacion):     {TMPR:.2f} horas')
print(f'TMEF (Tiempo Medio Entre Fallas):          {TMEF:.2f} horas (~{TMEF/24:.2f} dias)')
print('----------------------------------------------------------------')
print(f'Disponibilidad del Sistema (A):            {disponibilidad:.4f} %')
print(f'Indisponibilidad del Sistema (U):          {indisponibilidad:.4f} %')
print(f'Horas de inactividad estimadas al mes:     {horas_caida_mes:.2f} horas/mes')
print(f'Horas de inactividad estimadas al año:     {horas_caida_anio:.2f} horas/año')
print('----------------------------------------------------------------')
print(f'Acuerdo de Nivel de Servicio (SLA) Banca:  {SLA_BANCA:.4f} % (Max {horas_sla_anio:.2f} h/año)')
print(f'Brecha frente al SLA requerido:            {brecha_sla:.4f} puntos %')
print('----------------------------------------------------------------')
if disponibilidad >= SLA_BANCA:
    print('DICTAMEN FINAL: [CUMPLE] El sistema satisface los Acuerdos de Nivel de Servicio.')
else:
    print('DICTAMEN FINAL: [NO CUMPLE] El sistema NO ES APTO para despliegue bancario.')
    print(f'ADVERTENCIA: La inactividad mensual ({horas_caida_mes:.2f} h) excede {horas_caida_mes/(horas_sla_anio/12):.1f} veces el limite tolerable.')
print('================================================================')


---
## ACTIVIDAD 4: FORMULACIÓN DEL PLAN SEIS SIGMA (DMAMC) PARA EEI
**Objetivo:** Estructurar la hoja de ruta de mejora continua bajo el enfoque Seis Sigma para la causa raíz vital EEI (Especificaciones Erróneas / Incompletas).


In [ ]:
# ==============================================================================
# ACTIVIDAD 4: MATRIZ SEIS SIGMA DMAMC ESTRUCTURADA
# ==============================================================================
dmamc_data = {
    'Fase': ['Definir (D)', 'Medir (M)', 'Analizar (A)', 'Mejorar (M)', 'Controlar (C)'],
    'Objetivo SQA': [
        'Reducir en 50% los errores derivados de requerimientos ambiguos (de 21.8% a <= 10.9%) en 6 meses.',
        'Establecer la linea base cuantitativa de volatilidad de requisitos y conteo de TBA/TBD.',
        'Aislar las causas raiz que originan ambigüedad y desalineacion funcional.',
        'Implementar controles preventivos rigurosos y especificacion estructurada.',
        'Garantizar la sostenibilidad de las mejoras y evitar recurrencia de defectos.'
    ],
    'Accion Especifica': [
        'Delimitar modulos de transferencias y pagos; conformar equipo Black/Green Belt; elaborar Project Charter y SIPOC.',
        'Instrumentar conteo de TBA/TBD en backlog Jira; medir tasa de cambio por HU; calcular baseline DPMO (218,000).',
        'Aplicar diagrama de Ishikawa (6M), 5-Whys y Pareto secundario con Product Owners; validar con tabla de severidad.',
        'Institucionalizar RTF obligatorias en requerimientos; diseñar plantillas BDD (Given-When-Then); exigir Definition of Ready (DoR).',
        'Establecer auditorias quincenales de trazabilidad HU <-> Test Cases; implementar graficas de control SPC (UCL=2 cambios/HU).'
    ],
    'Entregable / KPI': [
        'Project Charter firmado | Matriz SIPOC',
        'Dashboard de Volatilidad | Baseline DPMO',
        'Diagrama Ishikawa validado | Top 3 causas raiz aisladas',
        'Plantilla HU v1.0 | Checklist DoR aprobada',
        'Auditorias quincenales | Graficas SPC en produccion'
    ]
}

df_dmamc = pd.DataFrame(dmamc_data)
display(df_dmamc)


---
## RESPUESTAS A LAS 5 PREGUNTAS DE ANÁLISIS TÉCNICO

### Pregunta 1: Análisis Pareto 80/20 —¿Cuál es la causa vital prioritaria y por qué?
La causa vital prioritaria es **EEI (Especificaciones Erróneas/Incompletas)** con **205 incidencias (21.8% del total)**. Las cuatro primeras categorías (EEI, Otros, MCC, ERD) acumulan el 72.7% de las incidencias, clasificándose como vitales bajo la regla de Pareto 80/20. Priorizar EEI es crítico porque concentra el 66.7% de los defectos Serios en la fase de análisis y tiene un efecto multiplicador preventivo: erradicar ambigüedades en requerimientos evita que se propaguen defectos hacia la comunicación (MCC), el diseño de datos (ERD) y la lógica de implementación (EDL).

### Pregunta 2: Confiabilidad y Disponibilidad frente al SLA bancario
El sistema **NO CUMPLE** con el SLA exigido para banca en línea (99.95%). Con un TMPF de 450 horas y un TMPR de 6 horas, la disponibilidad calculada es de **98.6750%**, generando una brecha negativa de **-1.2750 puntos porcentuales**. Esto equivale a **9.84 horas de caída al mes** (116.08 h/año), excediendo en 26.5 veces el máximo tolerable regulatorio (4.38 h/año). Se recomienda suspender el despliegue masivo y mantener la plataforma en Beta cerrada hasta automatizar el failover en microservicios reduciendo el TMPR a menos de 30 minutos.

### Pregunta 3: Enfoque Seis Sigma DMAMC —¿Por qué la fase Analizar es prioritaria?
La fase **Analizar** es prioritaria porque constituye el fundamento técnico del retorno de inversión (ROI). Sin un análisis causal riguroso mediante Ishikawa (6M) y 5-Whys, el equipo corre el riesgo de implementar herramientas costosas para atacar únicamente síntomas superficiales. La fase Analizar validó empíricamente que la raíz de EEI se debe a la ausencia de Definition of Ready (DoR), la falta de analistas funcionales en la captura y la carencia de criterios estructurados BDD.

### Pregunta 4: Clasificación según Norma ISO/IEC 25010
El 70% de las incidencias se concentran en dos características clave:
1. **Fiabilidad (40%):** Afecta la integridad de datos (ERD) y la tolerancia a fallos (EDL).
2. **Mantenibilidad (30%):** Afecta la modularidad (MCC) y el cumplimiento de estándares (VEP).
*Alerta crítica:* La ausencia de incidencias en **Seguridad** refleja una omisión de pruebas de penetración (DAST) en la fase Beta, no robustez del sistema, por lo que deben auditarse vulnerabilidades OWASP de inmediato.

### Pregunta 5: Recomendaciones para Optimizar el Proceso de Aseguramiento de Calidad
1. **Institucionalizar RTF en Requerimientos:** Revisiones técnicas formales con listas de chequeo antes de iniciar el desarrollo.
2. **Definition of Ready (DoR) Estricta:** Bloquear el ingreso de historias de usuario ambiguas o con elementos TBA/TBD.
3. **Criterios de Aceptación BDD (Given-When-Then):** Estandarizar reglas de negocio ejecutables.
4. **Dashboard Pareto Automatizado:** Monitoreo continuo de incidencias conectado a GitHub Issues.
5. **Auditorías Quincenales de Trazabilidad:** Garantizar el 100% de correlación entre requerimientos, casos de prueba y defectos.
